# UDA-Hub chat notebook

> **For reviewers**  
> This notebook preserves Udacity's starter chat example as a reference. The submitted UDA-Hub application is [03_agentic_app.py](03_agentic_app.py), which runs the LangGraph workflow in [agentic/workflow.py](agentic/workflow.py). The runnable cells below call that same application. See the [README demo](README.md#run-the-support-demo) for terminal commands.

## Prepare the demo

Open this notebook from the repository root and select the project's Python environment. Set up the synthetic databases with [01_external_db_setup.ipynb](01_external_db_setup.ipynb) and [02_core_db_setup.ipynb](02_core_db_setup.ipynb) if you haven't already. The examples run locally without an API key. Each invocation uses a fresh demo ticket and writes its normal local records and logs.

In [ ]:
import json
from contextlib import closing
from pathlib import Path
import sqlite3
import subprocess
import sys
from uuid import uuid4

root = Path.cwd()
app = root / "03_agentic_app.py"
cultpass_db = root / "data/external/cultpass.db"
core_db = root / "data/core/udahub.db"

if not app.is_file():
    raise FileNotFoundError("Set the notebook working directory to the repository root.")
if not (cultpass_db.is_file() and core_db.is_file()):
    raise FileNotFoundError("Run setup notebooks 01 and 02 before running this demo.")

def run_ticket(text, *options):
    command = [
        sys.executable, "-B", str(app),
        "--ticket-id", f"notebook-{uuid4().hex}", "--turn-id", "first", *options,
        "--", text,
    ]
    result = subprocess.run(command, cwd=root, capture_output=True, text=True, timeout=45)
    if result.returncode:
        raise RuntimeError(result.stdout.strip() or result.stderr.strip())
    print(json.dumps(json.loads(result.stdout), indent=2))

## Resolve a sign-in ticket

The answer cites the CultPass sign-in article. Edit the text to try another support request.

In [ ]:
run_ticket("I forgot my password. How can I sign in again?")

## Look up a subscription

Use a synthetic profile to get a personalized answer from CultPass account and subscription data.

In [ ]:
with closing(sqlite3.connect(f"{cultpass_db.as_uri()}?mode=ro", uri=True)) as db:
    profile = db.execute(
        "SELECT u.user_id FROM users u JOIN subscriptions s USING (user_id) "
        "WHERE u.is_blocked = 0 ORDER BY u.user_id LIMIT 1"
    ).fetchone()
if profile is None:
    raise RuntimeError("The CultPass demo database needs an active customer with a subscription.")

run_ticket(
    "What is my subscription status and monthly quota?",
    "--demo-customer-id", profile[0],
)

## Escalate a question outside the knowledge base

The workflow returns a handoff with an insufficient-evidence reason.

In [ ]:
run_ticket("How do I assemble a telescope?")

## Udacity starter chat (reference)

The original interactive example uses the [sample orchestrator](agentic/starter_workflow.py), a model credential, and a fixed thread. It is shown here as a code snippet so **Run All** can complete without interactive input.

```python
from dotenv import load_dotenv
from utils import chat_interface
load_dotenv()
from agentic.starter_workflow import orchestrator

chat_interface(orchestrator, "1")
list(orchestrator.get_state_history(
    config = {
        "configurable": {
            "thread_id": "1",
        }
    }
))[0].values["messages"]
```